# Tutoriel K-ABENA — Transformer / NLP (niveau 1 : notebook)
**Plateforme** : Hugging Face Transformers (PyTorch backend). Fine-tuning DistilBERT
sur classification de sentiments — **2 lignes** : remplacer `Trainer` par `KabenaTrainer`.
⚠ GPU fortement recommandé.

In [ ]:
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments)
from kabena.integrations.huggingface import KabenaTrainer   # <= LIGNE 1 (l'import)

ds = load_dataset("imdb", split={"train": "train[:2000]", "test": "test[:500]"})
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def prep(b): return tok(b["text"], truncation=True, padding="max_length", max_length=128)
ds = {k: v.map(prep, batched=True).rename_column("label", "labels") for k, v in ds.items()}

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
args = TrainingArguments(output_dir="out", num_train_epochs=1, per_device_train_batch_size=32,
                         optim="sgd", learning_rate=5e-3, report_to=[])   # SGD : cf. Limitation L1

trainer = KabenaTrainer(model=model, args=args, train_dataset=ds["train"],
                        eval_dataset=ds["test"], kabena_N=0.3, kabena_seed=0)
trainer.train()                                              # <= LIGNE 2
print(trainer.evaluate())

**Ce qui se passe sous le capot** : `KabenaTrainer.compute_loss` calcule la cross-entropy
par échantillon (`reduction='none'`), applique la sélection v3 + poids HT du batch, et
retourne la moyenne pondérée sur S*. Le gain porte sur le backward des exclues.
**Note L4** : non exécuté dans l'environnement de validation (GPU requis).